# 03 — Otimização de hiperparâmetros (nested CV)

**CMP263 (PPGC/UFRGS, 2026/2)** · Grupo: Artur Mariano, Emerson Herrera e Yaser Husein

**Pergunta:** depois de ajustar os hiperparâmetros, os algoritmos que lideraram o *spot-checking* (notebook 02) continuam empatados? Quanto cada um ganha com o ajuste?

Otimizamos os três líderes do *spot-checking* (SVR, Floresta e Gradient Boosting, que o Nemenyi não separou) e a **árvore**, que ficou em último com hiperparâmetros padrão por crescer sem poda. A avaliação usa **validação cruzada aninhada** (`nested_cv` de `src/enem_school/evaluation.py`), como na Aula 05:

| Laço | Partições | Papel |
|---|---|---|
| **Externo** | 5 dobras por município × 3 repetições (`assign_folds`), as mesmas do notebook 02 | estima o erro com escolas que não participaram da escolha |
| **Interno** | `GridSearchCV` com 3 dobras por município, só dentro do treino externo | escolhe os hiperparâmetros pelo RMSE |

O erro reportado mede o **procedimento completo** (algoritmo + busca em grade), não uma configuração escolhida olhando o teste.

## 1. Ambiente

A célula abaixo funciona **localmente** e **no Google Colab**:

- **Local:** rode a partir da raiz do repositório, depois de `uv sync --group notebooks`, e abra com `uv run jupyter lab`.
- **Colab:** a célula clona o repositório e instala as versões fixadas em `requirements.txt`. Se o Colab avisar que precisa reiniciar a sessão (por trocar a versão do pandas/numpy), reinicie e rode tudo de novo.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/emersonheto/enem-school-performance.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/enem-school-performance")
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Raiz do repositório: funciona com o notebook aberto em notebooks/ ou na raiz.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Raiz do repositório:", ROOT.name)
print("Colab:", IN_COLAB)

Raiz do repositório: enem-school-performance
Colab: False


In [2]:
import json
import random
import time

import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from enem_school import config
from enem_school.evaluation import compare_algorithms, nested_cv

random.seed(config.SEED)
np.random.seed(config.SEED)
pd.set_option("display.precision", 2)

RESULTS_DIR = config.REPORTS_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Dados

A tabela por escola é gerada por `scripts/build_dataset.py`. Se ela ainda não existir (por exemplo, numa sessão nova do Colab), a célula abaixo roda o script, que **baixa cerca de 1 GB do INEP** e leva alguns minutos.

Em seguida conferimos o SHA-256 dos arquivos brutos registrados nos metadados. Se o INEP republicar os microdados, os hashes mudam e a célula falha. Assim, nunca comparamos sem perceber resultados obtidos com dados diferentes.

In [3]:
if not config.SCHOOL_TABLE.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "build_dataset.py")], check=True, cwd=ROOT)

EXPECTED_SHA256 = {
    "enem": "b8d3e72cbf387ee08f18d552367005429017d43eea1f6d4b9bafe1d83a116071",
    "censo": "ae29d81d45ddecef59e11eda39788cbfd30dfa1cc71be9d29f23051fc2d640db",
}
metadata = json.loads(config.SCHOOL_TABLE_METADATA.read_text(encoding="utf-8"))
for source, expected in EXPECTED_SHA256.items():
    assert metadata["sources"][source]["sha256"] == expected, f"{source}: arquivo do INEP diferente do usado no trabalho"

schools = pd.read_parquet(config.SCHOOL_TABLE)
assert len(schools) == metadata["joined_schools"] == 23_374
print(f"{len(schools):,} escolas, {schools.shape[1]} colunas")
print(f"Alvo: média {schools[config.TARGET].mean():.1f}, desvio {schools[config.TARGET].std():.1f}")

23,374 escolas, 50 colunas
Alvo: média 524.7, desvio 62.4


## 3. Algoritmos e grades

Grades pequenas, com os hiperparâmetros que controlam a complexidade de cada modelo. O tamanho foi limitado pelo custo: cada combinação é treinada 3 (interno) × 15 (externo) = 45 vezes.

| Algoritmo | Hiperparâmetros | Combinações |
|---|---|---|
| **Árvore** | `max_depth` 5, 10, 15, 20 × `min_samples_leaf` 1, 10, 50 | 12 |
| **SVR** (alvo padronizado, como no notebook 02) | `C` 1, 3, 10 × `epsilon` 0,1, 0,3 | 6 |
| **Floresta** (100 árvores) | `max_depth` 10, 20, sem limite × `min_samples_leaf` 1, 5 | 6 |
| **Gradient Boosting** (300 árvores) | `learning_rate` 0,05, 0,1 × `max_depth` 3, 5 | 4 |

**Por que o `C` do SVR para em 10:** num treino de 12.500 escolas, `C=10` leva ≈ 10 s, mas `C=100` leva ≈ 140 s (o otimizador precisa ajustar quase todos os pontos e converge devagar). Com `C=100` a grade do SVR levaria horas.

As buscas rodam em paralelo (`n_jobs=-1`); por isso a Floresta usa `n_jobs=1` internamente, para não disputar os mesmos núcleos.

In [4]:
searches = {
    "Árvore": (
        DecisionTreeRegressor(random_state=config.SEED),
        {"max_depth": [5, 10, 15, 20], "min_samples_leaf": [1, 10, 50]},
    ),
    "SVR": (
        TransformedTargetRegressor(regressor=SVR(), transformer=StandardScaler()),
        {"regressor__C": [1, 3, 10], "regressor__epsilon": [0.1, 0.3]},
    ),
    "Floresta": (
        RandomForestRegressor(random_state=config.SEED, n_jobs=1),
        {"max_depth": [10, 20, None], "min_samples_leaf": [1, 5]},
    ),
    "Gradient Boosting": (
        GradientBoostingRegressor(n_estimators=300, random_state=config.SEED),
        {"learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    ),
}

## 4. Execução

Uma chamada de `nested_cv` por algoritmo. As partições externas são as mesmas para todos (e as mesmas do notebook 02), então a comparação continua **pareada**.

In [5]:
tuned = []
for name, (regressor, grid) in searches.items():
    start = time.perf_counter()
    tuned.append(nested_cv(regressor, grid, schools, variant="A", n_jobs=-1).assign(algoritmo=name))
    print(f"{name:<18} {(time.perf_counter() - start) / 60:5.1f} min")

tuned = pd.concat(tuned, ignore_index=True)
tuned.to_csv(RESULTS_DIR / "otimizacao.csv", index=False)

Árvore               0.2 min


SVR                  7.4 min


Floresta             4.2 min


Gradient Boosting    5.3 min


## 5. Padrão × ajustado

RMSE e R² médios nas 15 medições externas, com hiperparâmetros padrão (notebook 02) e ajustados (este notebook). As partições são as mesmas, então a diferença é efeito do ajuste.

In [6]:
default = pd.read_csv(RESULTS_DIR / "spot_checking_summary.csv", header=[0, 1], index_col=0)
summary = pd.DataFrame(
    {
        "RMSE padrão": default[("rmse", "mean")],
        "RMSE ajustado": tuned.groupby("algoritmo")["rmse"].mean(),
        "desvio ajustado": tuned.groupby("algoritmo")["rmse"].std(),
        "R² padrão": default[("r2", "mean")],
        "R² ajustado": tuned.groupby("algoritmo")["r2"].mean(),
    }
).loc[list(searches)].sort_values("RMSE ajustado")
summary["ganho (pontos)"] = summary["RMSE padrão"] - summary["RMSE ajustado"]
summary.to_csv(RESULTS_DIR / "otimizacao_summary.csv")
summary

,RMSE padrão,RMSE ajustado,desvio ajustado,R² padrão,R² ajustado,ganho (pontos)
algoritmo,,,,,,
Gradient Boosting,31.65,30.63,0.90,0.74,0.76,1.02
SVR,30.77,30.68,0.89,0.76,0.76,0.09
Floresta,30.99,30.91,0.89,0.75,0.75,0.08
Árvore,43.51,33.31,0.88,0.51,0.71,10.20


### Hiperparâmetros escolhidos

Quantas das 15 buscas internas escolheram cada combinação. Se a mesma combinação vence quase sempre, o ajuste é **estável**; se varia muito, o modelo é pouco sensível a esses hiperparâmetros (várias combinações empatam) ou a escolha depende dos municípios do treino.

In [7]:
tuned.assign(escolha=tuned["melhores_parametros"].astype(str)).groupby("algoritmo")["escolha"].value_counts().to_frame()

count
algoritmo         escolha                                              
Floresta          {'max_depth': 20, 'min_samples_leaf': 1}           12
                  {'max_depth': 20, 'min_samples_leaf': 5}            3
Gradient Boosting {'learning_rate': 0.05, 'max_depth': 5}            15
SVR               {'regressor__C': 1, 'regressor__epsilon': 0.3}     15
Árvore            {'max_depth': 10, 'min_samples_leaf': 10}          14
                  {'max_depth': 10, 'min_samples_leaf': 50}           1

## 6. Análise estatística

Mesmo procedimento do notebook 02 (Friedman + Nemenyi, Aula 08), agora sobre os modelos ajustados. Os mesmos cuidados valem: as 15 medições não são independentes, então os p-valores são otimistas.

In [8]:
comparison = compare_algorithms(tuned)
print(f"Friedman: χ² = {comparison['friedman_statistic']:.2f}, p = {comparison['friedman_p']:.2e}")
print(f"Nemenyi: CD = {comparison['critical_difference']:.3f} postos")
display(comparison["ranks"].to_frame())
comparison["significant_pairs"]

Friedman: χ² = 37.64, p = 3.37e-08
Nemenyi: CD = 1.211 postos


,posto médio
algoritmo,
Gradient Boosting,1.40
SVR,1.73
Floresta,2.87
Árvore,4.00


,A,B,diferença de postos
0,Gradient Boosting,Árvore,2.60
1,SVR,Árvore,2.27
2,Gradient Boosting,Floresta,1.47


## 7. Conclusões

- **Árvore:** o ajuste corrige o sobreajuste. O RMSE cai de 43,51 para 33,31 (ganho de 10,2 pontos) com `max_depth=10` em 15 de 15 buscas.
- **SVR e Floresta** quase não mudam (≈ 0,1 ponto): os padrões já estavam perto do melhor da grade. **Gradient Boosting** ganha 1,0 ponto (31,65 → 30,63) e passa a ter o menor RMSE.
- **Escolhas estáveis:** Gradient Boosting (`learning_rate=0.05`, `max_depth=5`) e SVR (`C=1`, `epsilon=0.3`) escolhem a mesma combinação nas 15 buscas; a Floresta escolhe `max_depth=20` nas 15.
- **Friedman** rejeita H0 (p ≈ 3e-8). Pelo **Nemenyi** (CD ≈ 1,21): Gradient Boosting e SVR são melhores que a árvore; Gradient Boosting é melhor que a Floresta; Gradient Boosting × SVR e SVR × Floresta **não se separam**.
- Entre os três ensembles/kernel a diferença é de **≤ 0,3 ponto de RMSE**: estatisticamente detectável em parte, mas pequena na escala do ENEM.
- Tempo total ≈ 17 min (SVR 7,4; Gradient Boosting 5,3; Floresta 4,2; árvore 0,2), num Mac com 12 núcleos.
- [ ] Decisão do grupo: qual modelo levar à interpretação (notebook 04). Os dados apontam **Gradient Boosting** (menor RMSE e melhor posto); SVR é equivalente em erro, mas mais difícil de interpretar.

**Limitações:** algumas escolhas ficaram **na borda da grade** (SVR: menor `C` e maior `epsilon`; Gradient Boosting: menor `learning_rate` e maior `max_depth`), então o ótimo pode estar fora dela; ampliar a grade aumentaria o custo. Os p-valores são otimistas pelos mesmos motivos do notebook 02.


## Ambiente de execução

Registro das versões usadas, para conferir com `uv.lock`.

In [9]:
import platform

import scipy
import sklearn

print("Python      ", platform.python_version())
for lib in (np, pd, scipy, sklearn):
    print(f"{lib.__name__:<12}", lib.__version__)
print("Plataforma  ", platform.platform())

Python       3.12.13
numpy        2.5.4
pandas       3.0.6
scipy        1.18.1
sklearn      1.9.1
Plataforma   macOS-27.2-arm64-arm-64bit
